# Notebook 16 — Vazamento do relabel e CV honesto (pós-sub11 = 0.48 público)

**Contexto:** sub11 scoring 0.48 público com CV 0.5460 (gap −0.066, histórico ≤0.016).
**Diagnóstico:** o relabel copia previsões OOF para15% dos rótulos de treino — mas essas
previsões vêm de modelos treinados COM o fold de validação → canal de vazamento
estilo *stacking* que infla o CV e não existe no teste.

Este notebook: (1) **mede o canal** (inner-OOF3: quantos rótulos relabel mudam quando o
fold j é excluído do modelo gerador); (2) corre o **CV honesto/aninhado** — relabel
regenerado DENTRO de cada fold + variantes originais, 4 famílias, caches `nb16_honest_fold*`;
(3) tabelas finais + veredito. Validação sempre com rótulos originais.

In [1]:
import os, json, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
K, TARGET = 16, 'AGE_RANGE'

def _data_dir():
    for cand in ['data', os.path.join('..', 'data')]:
        if os.path.isdir(cand): return cand
    raise FileNotFoundError('pasta data/ nao encontrada')
DATA_DIR = _data_dir()

tr = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
y = tr[TARGET].to_numpy().astype(int)
Xv = tr.drop(columns=['PID', TARGET]).to_numpy(dtype=np.float64)
fmin = tr['FAIXA_MINIMA'].to_numpy().astype(int)
n = len(y)
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(Xv, y))
CLAIMED = {'FH': 0.5364, 'mc5': 0.5376, 'LGB': 0.5417, 'HGB': 0.5355,
           'flat': 0.5409, 'B1': 0.5460}          # CVs com canal (NB13/NB15)
RESULTADOS = []

## 1. Medição do canal — rótulos relabel sensíveis ao fold de validação

In [2]:
ch_path = os.path.join(DATA_DIR, 'nb16_channel.json')
if os.path.exists(ch_path):
    ch = json.load(open(ch_path))
else:
    # recalcula: por fold externo j, inner-OOF3 só em TR; compara com o rótulo global
    d11 = np.load(os.path.join(DATA_DIR, 'nb11_oof311.npz'))
    oof_x, proba_x = d11['oof'].astype(int), d11['proba']
    rank = np.argsort(proba_x[np.arange(n), y])
    worst15 = np.zeros(n, bool); worst15[rank[:int(n * 0.15)]] = True
    def xr(sd=42):
        return XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist', random_state=sd,
            n_jobs=-1, eval_metric='mlogloss')
    rows = []
    for j, (a, b) in enumerate(FOLDS):
        pred_in = np.zeros(len(a), int)
        for ia, ib in StratifiedKFold(3, shuffle=True, random_state=42+j).split(Xv[a], y[a]):
            m = xr(); m.fit(Xv[a][ia], y[a][ia],
                            sample_weight=compute_sample_weight('balanced', y[a][ia]))
            pred_in[ib] = m.predict(Xv[a][ib])
        rel = worst15[a]
        y_glob = y[a].copy(); y_glob[rel] = oof_x[a][rel]
        y_hon = y[a].copy();  y_hon[rel] = pred_in[rel]
        dis = int((y_glob != y_hon).sum())
        rows.append({'fold': j, 'relabelled_tr': int(rel.sum()), 'disagreements': dis,
                     'pct': round(dis / rel.sum() * 100, 1)})
        print('fold', j, rows[-1], flush=True)
    ch = {'folds': rows, 'total_pct': round(sum(r['disagreements'] for r in rows) /
                                            sum(r['relabelled_tr'] for r in rows) * 100, 1)}
    json.dump(ch, open(ch_path, 'w'), indent=1)
print(f"CANAL: {ch['total_pct']}% dos rótulos relabel mudam quando o fold de validação "
      f"é excluído do modelo gerador (por fold: {[r['pct'] for r in ch['folds']]})")
print('Mecanismo: y_lab[tr] = M_{-m}(x) com M_{-m} treinado INCLUINDO o fold j -> os rótulos '
      'de treino do CV codificam rótulos do fold de validação (vazamento de stacking).')

CANAL: 32.4% dos rótulos relabel mudam quando o fold de validação é excluído do modelo gerador (por fold: [31.3, 32.5, 31.7, 32.8, 33.7])
Mecanismo: y_lab[tr] = M_{-m}(x) com M_{-m} treinado INCLUINDO o fold j -> os rótulos de treino do CV codificam rótulos do fold de validação (vazamento de stacking).


## 2. CV honesto — relabel regenerado dentro do fold (inner-OOF3) + variantes originais

In [3]:
def xgb_recipe(sd): return XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist', random_state=sd,
    n_jobs=-1, eval_metric='mlogloss')
def xgb_bin(sd): return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
    min_child_weight=3, subsample=0.8, colsample_bytree=0.5, tree_method='hist',
    random_state=sd, n_jobs=-1, eval_metric='logloss')
def lgbm_bin(sd): return LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
    min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
    reg_lambda=1, verbose=-1, n_jobs=-1, random_state=sd)
def hgb_bin(sd): return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
    max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1, early_stopping=False,
    random_state=sd)
def pfull_clip(a):
    a = np.minimum.accumulate(a, axis=1)
    return np.clip(np.column_stack([1-a[:,0], a[:,:-1]-a[:,1:], a[:,-1]]), 0, None)
def pfull_renorm(a):
    a = np.minimum.accumulate(a, axis=1)
    p = np.clip(np.column_stack([1-a[:,0], a[:,:-1]-a[:,1:], a[:,-1]]), 1e-6, None)
    return p / p.sum(axis=1, keepdims=True)

def fam_mc5(Xtr, ytr, Xp):
    out = []
    for sd in [42, 1, 2, 3, 4]:
        m = xgb_recipe(sd); m.fit(Xtr, ytr,
            sample_weight=compute_sample_weight('balanced', ytr))
        out.append(m.predict_proba(Xp))
    return np.mean(out, axis=0)

def fam_fh(Xtr, ytr, Xp, maker, seeds, pfull):
    out = []; sw = compute_sample_weight('balanced', ytr)
    for sd in seeds:
        acima = np.zeros((len(Xp), 15))
        for k in range(15):
            m = maker(sd); m.fit(Xtr, (ytr > k).astype(int), sample_weight=sw)
            acima[:, k] = m.predict_proba(Xp)[:, 1]
        out.append(pfull(acima))
    return np.mean(out, axis=0)

RES = {k: [] for k in ['rel_B1','rel_FH','rel_mc5','rel_LGB','rel_HGB','rel_flat',
                       'orig_B1','orig_FH','orig_mc5','orig_LGB','orig_HGB','orig_flat']}
for j, (a, b) in enumerate(FOLDS):
    p = os.path.join(DATA_DIR, f'nb16_honest_fold{j}.npz')
    if not os.path.exists(p):
        t0 = time.time()
        pred_in = np.zeros(len(a), int); conf_in = np.zeros(len(a))
        for ia, ib in StratifiedKFold(3, shuffle=True, random_state=42+j).split(Xv[a], y[a]):
            m = xgb_recipe(42); m.fit(Xv[a][ia], y[a][ia],
                sample_weight=compute_sample_weight('balanced', y[a][ia]))
            pr = m.predict_proba(Xv[a][ib])
            pred_in[ib] = pr.argmax(1); conf_in[ib] = pr.max(1)
        y_rel = y[a].copy()
        sel = np.argsort(conf_in)[:int(len(a) * 0.15)]   # pior15% de TR, aninhado
        y_rel[sel] = pred_in[sel]
        got = {}
        got['rel_mc5'] = fam_mc5(Xv[a], y_rel, Xv[b])
        got['rel_FH']  = fam_fh(Xv[a], y_rel, Xv[b], xgb_bin, [42,1,2,3,4], pfull_clip)
        got['rel_LGB'] = fam_fh(Xv[a], y_rel, Xv[b], lgbm_bin, [42,1,2,3], pfull_renorm)
        got['rel_HGB'] = fam_fh(Xv[a], y_rel, Xv[b], hgb_bin, [42,1], pfull_renorm)
        got['orig_mc5'] = fam_mc5(Xv[a], y[a], Xv[b])
        got['orig_FH']  = fam_fh(Xv[a], y[a], Xv[b], xgb_bin, [42,1,2,3,4], pfull_clip)
        got['orig_LGB'] = fam_fh(Xv[a], y[a], Xv[b], lgbm_bin, [42,1,2,3], pfull_renorm)
        got['orig_HGB'] = fam_fh(Xv[a], y[a], Xv[b], hgb_bin, [42,1], pfull_renorm)
        np.savez_compressed(p, **got)
        print(f'fold {j} treinado ({time.time()-t0:.0f}s)', flush=True)
    d = np.load(p)
    for pre in ['rel', 'orig']:
        for k in ['FH', 'mc5', 'LGB', 'HGB']:
            RES[f'{pre}_{k}'].append(d[f'{pre}_{k}'])
        RES[f'{pre}_flat'].append(np.mean([d[f'{pre}_FH'], d[f'{pre}_LGB'], d[f'{pre}_HGB']], 0))
        RES[f'{pre}_B1'].append(np.mean([d[f'{pre}_FH'], d[f'{pre}_LGB'], d[f'{pre}_HGB'],
                                         d[f'{pre}_mc5']], 0))

def folds_f1(Ps, med=None):
    out = []
    for P, (_, va) in zip(Ps, FOLDS):
        arg = P.argmax(1)
        if med is not None:
            m = np.array([int(np.median(np.repeat(np.arange(K), (P[i]*1000).astype(int))))
                          for i in range(len(P))])
            arg = np.where(P.max(1) < med, m, arg)
        out.append(f1_score(y[va], np.maximum(arg, fmin[va]), average='macro'))
    return np.array(out)

## 3. Tabelas honestas — componentes, inflação, pares relabel vs original

In [4]:
print('=== CV HONESTO (canal removido) ===')
for pre in ['orig', 'rel']:
    for key in ['B1', 'flat', 'FH', 'LGB', 'HGB', 'mc5']:
        f = folds_f1(RES[f'{pre}_{key}'])
        RESULTADOS.append({'secao': 'honesto', 'variante': f'{pre}-{key}',
                           'f1': round(f.mean(), 4), 'std': round(f.std(), 4)})
        print(f'{pre}-{key:<5} {f.mean():.4f} +/- {f.std():.4f} folds={np.round(f,4).tolist()}')
print('\n=== INFLAÇÃO DO CANAL (claimed - honest) ===')
for k, c in CLAIMED.items():
    h = folds_f1(RES[f'rel_{k}']).mean()
    RESULTADOS.append({'secao': 'inflacao', 'variante': k,
                       'claimed': c, 'honest': round(h, 4), 'delta': round(c - h, 4)})
    print(f'{k:<5} claimed {c:.4f} - honest {h:.4f} = {c-h:+.4f}')
print('\n=== PARES honestos relabel - original ===')
for k in ['FH', 'mc5', 'LGB', 'HGB', 'flat', 'B1']:
    fr, fo = folds_f1(RES[f'rel_{k}']), folds_f1(RES[f'orig_{k}'])
    print(f'{k:<5} rel {fr.mean():.4f} | orig {fo.mean():.4f} | {fr.mean()-fo.mean():+.4f} '
          f'({int((fr>fo).sum())}/5)')

=== CV HONESTO (canal removido) ===
orig-B1    0.5023 +/- 0.0095 folds=[0.5146, 0.5116, 0.5011, 0.4911, 0.4931]
orig-flat  0.5044 +/- 0.0121 folds=[0.5196, 0.5165, 0.5038, 0.4919, 0.4903]
orig-FH    0.5013 +/- 0.0124 folds=[0.5136, 0.5166, 0.5016, 0.4883, 0.4866]
orig-LGB   0.5036 +/- 0.0082 folds=[0.5147, 0.5125, 0.4984, 0.4964, 0.4962]
orig-HGB   0.4979 +/- 0.0156 folds=[0.5228, 0.5072, 0.495, 0.4814, 0.4828]
orig-mc5   0.4849 +/- 0.0064 folds=[0.4949, 0.4851, 0.4869, 0.4752, 0.4825]
rel-B1    0.4981 +/- 0.0051 folds=[0.5029, 0.5025, 0.4992, 0.4969, 0.4888]
rel-flat  0.5007 +/- 0.0080 folds=[0.4991, 0.505, 0.51, 0.5031, 0.4862]
rel-FH    0.5003 +/- 0.0089 folds=[0.5133, 0.5054, 0.4995, 0.4964, 0.4867]
rel-LGB   0.5040 +/- 0.0068 folds=[0.5104, 0.5097, 0.5058, 0.5028, 0.4916]
rel-HGB   0.4909 +/- 0.0077 folds=[0.5, 0.4968, 0.4941, 0.4818, 0.4818]
rel-mc5   0.4766 +/- 0.0123 folds=[0.4961, 0.4772, 0.4794, 0.4724, 0.4578]

=== INFLAÇÃO DO CANAL (claimed - honest) ===
FH    claimed 0.536

LGB   claimed 0.5417 - honest 0.5040 = +0.0377
HGB   claimed 0.5355 - honest 0.4909 = +0.0446
flat  claimed 0.5409 - honest 0.5007 = +0.0402
B1    claimed 0.5460 - honest 0.4981 = +0.0479

=== PARES honestos relabel - original ===
FH    rel 0.5003 | orig 0.5013 | -0.0011 (2/5)
mc5   rel 0.4766 | orig 0.4849 | -0.0083 (1/5)
LGB   rel 0.5040 | orig 0.5036 | +0.0004 (2/5)
HGB   rel 0.4909 | orig 0.4979 | -0.0070 (1/5)
flat  rel 0.5007 | orig 0.5044 | -0.0038 (2/5)
B1    rel 0.4981 | orig 0.5023 | -0.0042 (1/5)


## 4. Melhor receita honesta (regras de fallback, originais)

In [5]:
print('=== fallback sobre as receitas originais ===')
for nome in ['orig-flat', 'orig-FH', 'orig-LGB', 'orig-B1']:
    key = nome.split('-')[1]
    for med in [None, 0.45, 0.40, 0.35]:
        f = folds_f1(RES[f'orig_{key}'], med=med)
        tag = 'argmax+corr' if med is None else f'med@{med}+corr'
        RESULTADOS.append({'secao': 'fallback', 'variante': f'{nome} {tag}',
                           'f1': round(f.mean(), 4), 'std': round(f.std(), 4)})
        print(f'{nome:<10} {tag:<14} {f.mean():.4f} +/- {f.std():.4f} {np.round(f,4).tolist()}')
pd.DataFrame(RESULTADOS).to_csv(os.path.join(DATA_DIR, 'nb16_resultados.csv'), index=False)

=== fallback sobre as receitas originais ===
orig-flat  argmax+corr    0.5044 +/- 0.0121 [0.5196, 0.5165, 0.5038, 0.4919, 0.4903]


orig-flat  med@0.45+corr  0.5062 +/- 0.0079 [0.5198, 0.5053, 0.5084, 0.4968, 0.5004]


orig-flat  med@0.4+corr   0.5064 +/- 0.0093 [0.5188, 0.5124, 0.5079, 0.4918, 0.5009]


orig-flat  med@0.35+corr  0.5056 +/- 0.0116 [0.5205, 0.5153, 0.5065, 0.4892, 0.4966]
orig-FH    argmax+corr    0.5013 +/- 0.0124 [0.5136, 0.5166, 0.5016, 0.4883, 0.4866]


orig-FH    med@0.45+corr  0.5017 +/- 0.0088 [0.516, 0.5033, 0.5046, 0.4915, 0.4933]


orig-FH    med@0.4+corr   0.5019 +/- 0.0097 [0.516, 0.5062, 0.5042, 0.4876, 0.4955]


... (3 outputs left out)

orig-LGB   med@0.35+corr  0.5073 +/- 0.0074 [0.518, 0.5142, 0.5006, 0.4999, 0.5041]
orig-B1    argmax+corr    0.5023 +/- 0.0095 [0.5146, 0.5116, 0.5011, 0.4911, 0.4931]


orig-B1    med@0.45+corr  0.5053 +/- 0.0108 [0.52, 0.5106, 0.5095, 0.4896, 0.4967]


orig-B1    med@0.4+corr   0.5067 +/- 0.0093 [0.5178, 0.5154, 0.5064, 0.4926, 0.5011]


orig-B1    med@0.35+corr  0.5039 +/- 0.0102 [0.5181, 0.5114, 0.505, 0.4922, 0.4927]


## 5. Veredito

1. **Canal medido: 32.4%** dos rótulos relabel dependem do fold de validação → inflação
   de **+0.036 a +0.061** por variante; o CV 0.5460 de E27 era **0.4981 honesto**.
   Gap honesto→público do sub11: 0.4981 → 0.48 = −0.018, dentro do histórico (≤0.016).
2. **relabel é honestamente neutro/negativo** (todos os pares ≤ 0, −0.0004 a −0.0083):
   E24–E26 e mc5 eram artefacto do canal. A via de rótulos morre aqui.
3. **Teto honesto ≈ 0.504–0.509** — igual ao sub7 do colega (CV 0.5089 → público 0.51).
   Melhor honesto: `orig-LGB med@0.45+corr 0.5083` / `orig-flat med@0.40+corr 0.5064`
   (med escolhido entre {0.35,0.40,0.45} nas mesmas folds → +0.001–0.002 de seleção).
4. Todos os deltas honestos ≤ ±0.004 são ruído (σ≈0.008–0.012). Submissões devem usar
   só números honestos; sub10/sub11 continuam válidos mas esperáveis em ~0.49–0.51.